# Assignment 3: Human Activity Recognition (HAR)

Predict activity label (0–5) from wrist-worn accelerometer data aggregated into 1-second intervals over 5-minute windows.

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.metrics import f1_score, classification_report, confusion_matrix

import model.utils as utils

sns.set_style("whitegrid")
%matplotlib inline

## 1. Load Data

Aggregate each CSV (300 time steps) into per-file statistical features.

In [ ]:
DATA_DIR = "data"

X_train, y_train, train_ids, train_users = utils.load_train_data(f"{DATA_DIR}/train/train")
X_test, test_ids, test_users = utils.load_test_data(f"{DATA_DIR}/test/test")

print(f"Train: {X_train.shape[0]} samples, {X_train.shape[1]} features")
print(f"Test:  {X_test.shape[0]} samples, {X_test.shape[1]} features")

## 2. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

counts = y_train.value_counts().sort_index()
axes[0].bar(counts.index.astype(str), counts.values,
           color=sns.color_palette("Set2", 6))
axes[0].set_title("Label Distribution (Train)")
axes[0].set_xlabel("Activity Label")
axes[0].set_ylabel("Count")

axes[1].pie(counts.values, labels=counts.index,
           autopct='%1.1f%%', colors=sns.color_palette("Set2", 6))
axes[1].set_title("Label Proportions")

plt.tight_layout()
plt.show()

print("\nClass counts:")
print(counts)

In [ ]:
mean_features = ["mean_x__mean", "mean_y__mean", "mean_z__mean",
                "std_x__mean", "std_y__mean", "std_z__mean"]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, feat in enumerate(mean_features):
    for label in range(6):
        mask = y_train == label
        axes[i].hist(X_train.loc[mask, feat], bins=40, alpha=0.5,
                    label=f"Class {label}", density=True)
    axes[i].set_title(feat)
    axes[i].legend(fontsize=7, loc="upper right")

plt.suptitle("Per-Class Feature Distributions", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
corr = X_train.corr()

fig, ax = plt.subplots(figsize=(18, 16))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap="RdBu_r", center=0,
            square=True, linewidths=0.5, ax=ax,
            cbar_kws={"shrink": 0.6})
ax.set_title("Feature Correlation Matrix", fontsize=14)
plt.tight_layout()
plt.show()

**Key observations:**
- Classes 0 and 1 dominate (~84% combined). Classes 2–5 are severely underrepresented.
- mean_z features show clear separation between activities.
- Many features are correlated (especially mean/std derivatives of the same axis).

## 3. Baseline Model

Random Forest with `class_weight='balanced'` and user-based GroupKFold (5 splits).
GroupKFold ensures all samples from the same user stay in the same fold, preventing data leakage.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_train)

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

gkf = GroupKFold(n_splits=5)
scores = cross_val_score(
    model, X_scaled, y_train,
    cv=gkf, groups=train_users,
    scoring="f1_macro", n_jobs=-1
)

for i, s in enumerate(scores, 1):
    print(f"Fold {i}: F1-macro = {s:.4f}")
print(f"\nMean F1-macro: {scores.mean():.4f}  (+/- {scores.std():.4f})")

In [ ]:
from sklearn.model_selection import cross_val_predict

y_pred_cv = cross_val_predict(model, X_scaled, y_train,
                              cv=gkf, groups=train_users, n_jobs=-1)

print(classification_report(y_train, y_pred_cv, digits=4))

fig, ax = plt.subplots(figsize=(7, 6))
cm = confusion_matrix(y_train, y_pred_cv, normalize="true")
sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues", ax=ax,
            xticklabels=range(6), yticklabels=range(6))
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Confusion Matrix (Row-Normalized)")
plt.show()

## 3b. XGBoost + LightGBM Tuning (Optuna)

Optuna searches for best hyperparameters using GroupKFold-5 (user-based splits).

In [ ]:
from model.train import tune_xgboost, tune_lightgbm, cv_evaluate
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

print("=== Tuning XGBoost ===")
xgb_params, xgb_model = tune_xgboost(X_train, y_train, train_users, n_trials=50)

print("Best XGBoost params:", xgb_params)
xgb_scores, xgb_mean, xgb_std = cv_evaluate(xgb_model, X_train, y_train, train_users)
print(f"XGBoost CV F1-macro: {xgb_mean:.4f} (+/- {xgb_std:.4f})")
for i, s in enumerate(xgb_scores, 1):
    print(f"  Fold {i}: {s:.4f}")

In [ ]:
print("\n=== Tuning LightGBM ===")
lgb_params, lgb_model = tune_lightgbm(X_train, y_train, train_users, n_trials=50)

print("Best LightGBM params:", lgb_params)
lgb_scores, lgb_mean, lgb_std = cv_evaluate(lgb_model, X_train, y_train, train_users)
print(f"LightGBM CV F1-macro: {lgb_mean:.4f} (+/- {lgb_std:.4f})")
for i, s in enumerate(lgb_scores, 1):
    print(f"  Fold {i}: {s:.4f}")

## 3c. Ensemble Prediction & Submission

Average  from both models, then . Also save individual model predictions for comparison.

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler
import model.utils as utils
import os

os.makedirs("output", exist_ok=True)

# Scale full training data for fitting
scaler_full = StandardScaler()
X_train_scaled = scaler_full.fit_transform(X_train)

xgb_model.fit(X_train_scaled, y_train)
lgb_model.fit(X_train_scaled, y_train)

# Scale test data
X_test_scaled = scaler_full.transform(X_test)

# XGBoost predictions
xgb_preds = xgb_model.predict(X_test_scaled)
utils.generate_submission(test_ids, xgb_preds, "output/submission_xgb.csv")

# LightGBM predictions
lgb_preds = lgb_model.predict(X_test_scaled)
utils.generate_submission(test_ids, lgb_preds, "output/submission_lgb.csv")

# Ensemble: average probabilities
xgb_proba = xgb_model.predict_proba(X_test_scaled)
lgb_proba = lgb_model.predict_proba(X_test_scaled)
ensemble_proba = (xgb_proba + lgb_proba) / 2
ensemble_preds = np.argmax(ensemble_proba, axis=1)
utils.generate_submission(test_ids, ensemble_preds, "output/submission_ensemble.csv")

print("Saved:")
print("  output/submission_xgb.csv")
print("  output/submission_lgb.csv")
print("  output/submission_ensemble.csv")

## 4. Test Prediction & Submission

In [ ]:
model.fit(X_scaled, y_train)

X_test_scaled = scaler.transform(X_test)
test_preds = model.predict(X_test_scaled)

utils.generate_submission(test_ids, test_preds, "output/submission_baseline_rf.csv")

## 5. Improvement Directions

Current baseline aggregating 300 time steps into global statistics discards temporal structure.

**Ideas to explore:**

| Direction | Description |
|---|---|
| Temporal models | LSTM / GRU / 1D-CNN over raw 300-step sequences |
| More features | FFT coefficients, rolling window stats, jerk (derivative), signal magnitude area |
| Class imbalance | SMOTE, ADASYN, or custom sampler for minority classes |
| Feature selection | RFE, mutual information to drop redundant 42 features |
| Ensemble | XGBoost / LightGBM with tuned hyperparams + bagging |
| User normalization | Per-user feature normalization to reduce inter-subject variability |
| Window splitting | Treat each CSV as 5 x 1-min windows for more training samples |